In [ ]:
import torch
print(torch.cuda.get_device_name(0))

Tesla T4


In [ ]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes trl python-dotenv sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 12.9 MB/s eta 0:00:00


In [ ]:
!git clone https://github.com/karad-tanmay/sgft-demo.git

Cloning into 'sgft-demo'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 28 (delta 2), reused 26 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 458.75 KiB | 7.28 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [ ]:
cd sgft-demo/

/content/sgft-demo


In [ ]:
pwd

'/content/sgft-demo'

In [ ]:
ls

'=0.46.1'   data/            README.md          src/
 configs/   env_sample.txt   requirements.txt


In [ ]:
with open(".env", "w") as f:
    f.write("""MODEL_NAME=Qwen/Qwen2.5-0.5B-Instruct
OUTPUT_DIR=outputs/qwen25_05b
DATA_PATH=data/processed/sg_dataset.json
BATCH_SIZE=1
GRAD_ACCUM=8
EPOCHS=3
LR=2e-4
MAX_LENGTH=256
SEED=42
""")

print(open(".env").read())

MODEL_NAME=Qwen/Qwen2.5-0.5B-Instruct
OUTPUT_DIR=outputs/qwen25_05b
DATA_PATH=data/processed/sg_dataset.json
BATCH_SIZE=1
GRAD_ACCUM=8
EPOCHS=3
LR=2e-4
MAX_LENGTH=256
SEED=42



In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(".env")

MODEL_NAME = os.getenv("MODEL_NAME")
OUTPUT_DIR = os.getenv("OUTPUT_DIR")
DATA_PATH = os.getenv("DATA_PATH")
BATCH_SIZE = int(os.getenv("BATCH_SIZE"))
GRAD_ACCUM = int(os.getenv("GRAD_ACCUM"))
EPOCHS = int(os.getenv("EPOCHS"))
LR = float(os.getenv("LR"))
MAX_LENGTH = int(os.getenv("MAX_LENGTH"))
SEED = int(os.getenv("SEED"))

print("MODEL_NAME =", MODEL_NAME)
print("OUTPUT_DIR =", OUTPUT_DIR)
print("DATA_PATH =", DATA_PATH)
print("BATCH_SIZE =", BATCH_SIZE)
print("GRAD_ACCUM =", GRAD_ACCUM)
print("EPOCHS =", EPOCHS)
print("LR =", LR)
print("MAX_LENGTH =", MAX_LENGTH)
print("SEED =", SEED)

MODEL_NAME = Qwen/Qwen2.5-0.5B-Instruct
OUTPUT_DIR = outputs/qwen25_05b
DATA_PATH = data/processed/sg_dataset.json
BATCH_SIZE = 1
GRAD_ACCUM = 8
EPOCHS = 3
LR = 0.0002
MAX_LENGTH = 256
SEED = 42


In [ ]:
!pip uninstall -y pyarrow datasets
!pip install -U pyarrow datasets

Found existing installation: pyarrow 23.0.1
Uninstalling pyarrow-23.0.1:
  Successfully uninstalled pyarrow-23.0.1
Found existing installation: datasets 4.8.4
Uninstalling datasets-4.8.4:
  Successfully uninstalled datasets-4.8.4
  Using cached pyarrow-23.0.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.1 kB)
  Using cached datasets-4.8.4-py3-none-any.whl.metadata (19 kB)
Using cached pyarrow-23.0.1-cp312-cp312-manylinux_2_28_x86_64.whl (47.6 MB)
Using cached datasets-4.8.4-py3-none-any.whl (526 kB)


In [ ]:
import json
from datasets import Dataset

with open("data/processed/sg_dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total raw samples:", len(data))
print("First sample:", data[0])

dataset = Dataset.from_list(data)
dataset = dataset.train_test_split(test_size=0.15, seed=SEED)

print(dataset)
print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["test"]))

Total raw samples: 3000
First sample: {'input': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'output': "Here's step-by-step solution guidance:\n\n1.  Identify the number of clips Natalia sold in April.\n2.  Determine how to calculate the number of clips sold in May based on the information given about April's sales.\n3.  Combine the number of clips sold in April and the number of clips sold in May to find the total."}
DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 2550
    })
    test: Dataset({
        features: ['input', 'output'],
        num_rows: 450
    })
})
Train size: 2550
Validation size: 450


In [ ]:
def format_example(example):
    return {
        "text": f"Question: {example['input']}\n\nSolution:\n{example['output']}"
    }

dataset = dataset.map(format_example)

print(dataset["train"][0]["text"])

Map:   0%|          | 0/2550 [00:00<?, ? examples/s]

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Question: Frank is walking through a corn maze. He has already spent 45 minutes inside. He has done 4 other corn mazes and finished those in 50 minutes on average. How much longer can he spend inside if he wants to ensure that his average doesn't go above 60 minutes per maze?

Solution:
Step 1: Calculate the total time spent across all previous mazes combined.
Step 2: Determine the total allowed time for all mazes based on the desired average duration.
Step 3: Subtract the total time already spent on all mazes from the total allowed time to find the remaining budget.
Step 4: Subtract the time already spent in the current maze from the remaining budget to find how much longer he can stay.


In [ ]:
from datasets import Dataset

seen = set()
unique_rows = []

for ex in dataset["train"]:
    if ex["text"] not in seen:
        seen.add(ex["text"])
        unique_rows.append(ex)

dataset["train"] = Dataset.from_list(unique_rows)

print("Train size after dedup:", len(dataset["train"]))
print("Validation size:", len(dataset["test"]))

Train size after dedup: 2550
Validation size: 450


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Pad token: <|endoftext|>
EOS token: <|im_end|>


In [ ]:
lengths = []

check_count = min(200, len(dataset["train"]))
for i in range(check_count):
    txt = dataset["train"][i]["text"]
    lengths.append(len(tokenizer(txt, truncation=False)["input_ids"]))

print("Checked:", check_count)
print("Min tokens:", min(lengths))
print("Max tokens:", max(lengths))
print("Avg tokens:", sum(lengths) / len(lengths))

Checked: 200
Min tokens: 65
Max tokens: 269
Avg tokens: 131.84


In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
    trust_remote_code=True
)

model.config.use_cache = False

print("Model loaded.")
print("Model dtype:", next(model.parameters()).dtype)

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.
Model dtype: torch.float16


In [ ]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
print("Model prepared for LoRA training.")

Model prepared for LoRA training.


In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=False,
    bf16=True,
    report_to="none",
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_grad_norm=1.0,
    seed=SEED,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    dataset_text_field="text",
    max_length=MAX_LENGTH,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=training_args,
    processing_class=tokenizer,
)

print("Trainer ready.")

Adding EOS to train dataset:   0%|          | 0/2550 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2550 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/450 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/450 [00:00<?, ? examples/s]

Trainer ready.


In [ ]:
train_result = trainer.train()
print(train_result)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss
1,0.814833,0.818115
2,0.693568,0.797768
3,0.613508,0.807175


TrainOutput(global_step=957, training_loss=0.7402262418621386, metrics={'train_runtime': 3984.669, 'train_samples_per_second': 1.92, 'train_steps_per_second': 0.24, 'total_flos': 2154153751200000.0, 'train_loss': 0.7402262418621386})


In [ ]:
import torch
print(torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())

Tesla T4
BF16 supported: True


In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved to:", OUTPUT_DIR)

Saved to: outputs/qwen25_05b


In [ ]:
import os
import json

metrics_path = os.path.join(OUTPUT_DIR, "train_metrics.json")

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(train_result.metrics, f, indent=2)

print("Metrics saved to:", metrics_path)
print(train_result.metrics)

Metrics saved to: outputs/qwen25_05b/train_metrics.json
{'train_runtime': 3984.669, 'train_samples_per_second': 1.92, 'train_steps_per_second': 0.24, 'total_flos': 2154153751200000.0, 'train_loss': 0.7402262418621386}


In [ ]:
import torch

questions = [
    "What is 12 + 25?",
    "A number is multiplied by 3 and then 5 is added to get 20. What is the number?",
    "Solve step by step: 2x + 5 = 13",
    "If a train travels 60 km in 1 hour, how far will it travel in 5 hours?"
]

for i, q in enumerate(questions, 1):
    prompt = f"""Question: {q}

Answer in this format:
Reasoning: show the steps clearly.
Final Answer: give only the final result.

Response:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f"\n{'='*60}")
    print(f"Question {i}: {q}")
    print(f"{'='*60}")
    print(response)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Question 1: What is 12 + 25?
Question: What is 12 + 25?

Answer in this format:
Reasoning: show the steps clearly.
Final Answer: give only the final result.

Response:
Step 1: Identify the two numbers being added together.
Step 2: Combine the first number with the second number to find their sum.

Question 2: A number is multiplied by 3 and then 5 is added to get 20. What is the number?
Question: A number is multiplied by 3 and then 5 is added to get 20. What is the number?

Answer in this format:
Reasoning: show the steps clearly.
Final Answer: give only the final result.

Response:
Step 1: Identify the mathematical operation required to find the unknown value.
Step 2: Determine the relationship between the known quantity and the resulting sum.
Step 3: Apply the given multiplier to the known value to isolate the missing term.
Step 4: Combine the isolated term with the addition of the constant to find the original value.

Question 3: Solve step by step: 2x + 5 = 13
Question: Solve ste

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# CHANGE THIS to your base model
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

# Where everything will be saved
SAVE_PATH = "/content/drive/MyDrive/outputs/qwen25_05b"

import os
os.makedirs(SAVE_PATH, exist_ok=True)

print("Save path ready:", SAVE_PATH)

Save path ready: /content/drive/MyDrive/outputs/qwen25_05b


In [ ]:
# 1) See what is currently inside your local saved model folder
!ls -R {OUTPUT_DIR}

outputs/qwen25_05b:
adapter_config.json	   checkpoint-638	  tokenizer.json
adapter_model.safetensors  checkpoint-957	  training_args.bin
chat_template.jinja	   README.md		  train_metrics.json
checkpoint-319		   tokenizer_config.json

outputs/qwen25_05b/checkpoint-319:
adapter_config.json	   README.md		  tokenizer.json
adapter_model.safetensors  rng_state.pth	  trainer_state.json
chat_template.jinja	   scheduler.pt		  training_args.bin
optimizer.pt		   tokenizer_config.json

outputs/qwen25_05b/checkpoint-638:
adapter_config.json	   README.md		  tokenizer.json
adapter_model.safetensors  rng_state.pth	  trainer_state.json
chat_template.jinja	   scheduler.pt		  training_args.bin
optimizer.pt		   tokenizer_config.json

outputs/qwen25_05b/checkpoint-957:
adapter_config.json	   README.md		  tokenizer.json
adapter_model.safetensors  rng_state.pth	  trainer_state.json
chat_template.jinja	   scheduler.pt		  training_args.bin
optimizer.pt		   tokenizer_config.json


In [ ]:
# 2) Copy the entire saved folder from Colab local storage to Google Drive
import shutil
import os

if os.path.exists(SAVE_PATH):
    print("Drive folder exists:", SAVE_PATH)
else:
    os.makedirs(SAVE_PATH, exist_ok=True)

for item in os.listdir(OUTPUT_DIR):
    src = os.path.join(OUTPUT_DIR, item)
    dst = os.path.join(SAVE_PATH, item)

    if os.path.isdir(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)

print("Copied all files from OUTPUT_DIR to SAVE_PATH")

Drive folder exists: /content/drive/MyDrive/outputs/qwen25_05b
Copied all files from OUTPUT_DIR to SAVE_PATH


In [ ]:
# 3) Save the base model name into Drive
with open(f"{SAVE_PATH}/base_model.txt", "w") as f:
    f.write(MODEL_NAME)

print("Saved base_model.txt")

Saved base_model.txt


In [ ]:
# 4) If trainer exists, also save training args to Drive
import torch

torch.save(trainer.args, f"{SAVE_PATH}/training_args.bin")
print("Saved training_args.bin")

Saved training_args.bin


In [ ]:
# 5) Verify final files in Drive
!ls -R {SAVE_PATH}

/content/drive/MyDrive/outputs/qwen25_05b:
adapter_config.json	   checkpoint-319  tokenizer_config.json
adapter_model.safetensors  checkpoint-638  tokenizer.json
base_model.txt		   checkpoint-957  training_args.bin
chat_template.jinja	   README.md	   train_metrics.json

/content/drive/MyDrive/outputs/qwen25_05b/checkpoint-319:
adapter_config.json	   README.md		  tokenizer.json
adapter_model.safetensors  rng_state.pth	  trainer_state.json
chat_template.jinja	   scheduler.pt		  training_args.bin
optimizer.pt		   tokenizer_config.json

/content/drive/MyDrive/outputs/qwen25_05b/checkpoint-638:
adapter_config.json	   README.md		  tokenizer.json
adapter_model.safetensors  rng_state.pth	  trainer_state.json
chat_template.jinja	   scheduler.pt		  training_args.bin
optimizer.pt		   tokenizer_config.json

/content/drive/MyDrive/outputs/qwen25_05b/checkpoint-957:
adapter_config.json	   README.md		  tokenizer.json
adapter_model.safetensors  rng_state.pth	  trainer_state.json
chat_template.jinja	  

In [ ]:
# 6) Optional zip backup
!zip -r /content/drive/MyDrive/outputs/qwen25_05b.zip {SAVE_PATH}

  adding: content/drive/MyDrive/outputs/qwen25_05b/ (stored 0%)
  adding: content/drive/MyDrive/outputs/qwen25_05b/checkpoint-638/ (stored 0%)
  adding: content/drive/MyDrive/outputs/qwen25_05b/checkpoint-638/trainer_state.json (deflated 77%)
  adding: content/drive/MyDrive/outputs/qwen25_05b/checkpoint-638/adapter_model.safetensors (deflated 21%)
  adding: content/drive/MyDrive/outputs/qwen25_05b/checkpoint-638/scheduler.pt (deflated 61%)
  adding: content/drive/MyDrive/outputs/qwen25_05b/checkpoint-638/chat_template.jinja (deflated 71%)
  adding: content/drive/MyDrive/outputs/qwen25_05b/checkpoint-638/optimizer.pt (deflated 22%)
  adding: content/drive/MyDrive/outputs/qwen25_05b/checkpoint-638/tokenizer.json (deflated 81%)
  adding: content/drive/MyDrive/outputs/qwen25_05b/checkpoint-638/rng_state.pth (deflated 26%)
  adding: content/drive/MyDrive/outputs/qwen25_05b/checkpoint-638/training_args.bin (deflated 52%)
  adding: content/drive/MyDrive/outputs/qwen25_05b/checkpoint-638/adapt